Problem 1 – Row-wise ℓ₂ normalization
-------------------------------------
Given a float tensor **X** of shape `(B, D)`, return a tensor **Y**
with the same shape such that every row of **Y** has (approximate)
ℓ₂-norm == 1.

Mathematically  
 Yᵢ = Xᵢ / (‖Xᵢ‖₂ + ε)

Constraints
-----------
* 1 ≤ B·D ≤ 10⁷
* Use **no Python for-loops**.

Example
-------
```python
X = torch.tensor([[3., 4.]])
Y = normalize_rows(X)
Y.norm(dim=1)

>>> tensor([1.])
```

In [4]:
import torch

def normalize_rows(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return X / (torch.linalg.vector_norm(X, ord=2, dim=-1, keepdim=True) + eps)


# ---------- reference + tests (do not modify) ----------
def _ref_normalize_rows(X, eps=1e-8):
    return X / (X.norm(dim=1, keepdim=True) + eps)

def _self_check():
    X = torch.randn(5, 7)
    try:
        Y = normalize_rows(X)
    except NotImplementedError:
        print("🔧  Implement normalize_rows and re-run the cell.")
        return
    assert Y.shape == X.shape, "Shape mismatch"
    assert torch.allclose(Y, _ref_normalize_rows(X)), "Wrong answer"
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 2 – Channel-wise z-score
--------------------------------
For an image batch tensor **imgs** of shape `(N, C, H, W)` compute

    (img − μ_c) / (σ_c + ε)

where μ_c and σ_c are the mean and std of channel *c* over the
(N, H, W) axes.

Return a tensor of the same shape.

No Python loops allowed.

In [10]:
import torch

def channel_zscore(imgs: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    N, C, H, W = imgs.shape
    tmp_imgs = imgs.permute(1, 0, 2, 3).reshape(C, N*H*W)
    avg = tmp_imgs.mean(dim=-1).reshape(1, C, 1, 1)
    std = tmp_imgs.std(dim=-1).reshape(1, C, 1, 1)
    return (imgs - avg) / (std + eps)

# reference & test
def _ref_channel_zscore(imgs, eps=1e-5):
    mean = imgs.mean(dim=(0, 2, 3), keepdim=True)
    std  = imgs.std (dim=(0, 2, 3), keepdim=True) + eps
    return (imgs - mean) / std

def _self_check():
    imgs = torch.randn(4, 3, 16, 16)
    try:
        out = channel_zscore(imgs)
    except NotImplementedError:
        print("🔧  Implement channel_zscore.")
        return
    assert torch.allclose(out, _ref_channel_zscore(imgs), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 3 – Numerically stable softmax per row
----------------------------------------------
Given a tensor **S** of shape `(B, K)`, return

    softmax(Sᵢ)  for each row i,

implemented with the usual “shift by max” trick to avoid overflow.

In [13]:
import torch

def stable_row_softmax(S: torch.Tensor) -> torch.Tensor:
    B, K = S.shape
    S = S - S.max(dim=-1, keepdim=True).values
    return S.exp() / S.exp().sum(dim=-1, keepdim=True)


def _ref_stable_row_softmax(S):
    Z = S - S.max(dim=1, keepdim=True).values
    eZ = Z.exp()
    return eZ / eZ.sum(dim=1, keepdim=True)

def _self_check():
    S = torch.randn(3, 9) * 25
    try:
        out = stable_row_softmax(S)
    except NotImplementedError:
        print("🔧  Implement stable_row_softmax.")
        return
    assert torch.allclose(out, _ref_stable_row_softmax(S), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 4 – Masked fill
-----------------------
Given
* **logits** of arbitrary shape `(*)`
* a boolean **mask** of the same shape  
return a tensor where entries with `mask == False` are replaced by
−∞ (so they will become zero after softmax).

In [14]:
import torch

def masked_logits(logits: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill_(~mask, -torch.inf)


def _ref_masked_logits(logits, mask):
    return logits.masked_fill(~mask, float("-inf"))

def _self_check():
    L = torch.randn(2, 5)
    M = torch.rand(2, 5) > .4
    try:
        out = masked_logits(L, M)
    except NotImplementedError:
        print("🔧  Implement masked_logits.")
        return
    assert torch.isneginf(out[~M]).all()
    assert torch.equal(out, _ref_masked_logits(L, M))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 5 – Batched cosine
--------------------------
Input
* **A** ∈ ℝ^{B×M×D}
* **B** ∈ ℝ^{B×N×D}

Output **S** ∈ ℝ^{B×M×N} where  
 S[b, i, j] = cosine_similarity(A[b,i], B[b,j]).

In [15]:
import torch

def batched_cosine(A: torch.Tensor, B: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    b, m, d = A.shape
    b, n, d = B.shape

    P = torch.einsum("bmd,bnd->bmn", A, B)
    normA = torch.linalg.vector_norm(A, dim=-1, keepdim=True)
    normB = torch.linalg.vector_norm(B, dim=-1, keepdim=True).view(b, 1, n)

    return P / (normA * normB + eps)


def _ref_batched_cosine(A, B, eps=1e-8):
    A = A / (A.norm(dim=-1, keepdim=True) + eps)
    B = B / (B.norm(dim=-1, keepdim=True) + eps)
    return torch.einsum("bmd,bnd->bmn", A, B)

def _self_check():
    A, B = torch.randn(2, 3, 4), torch.randn(2, 5, 4)
    try:
        out = batched_cosine(A, B)
    except NotImplementedError:
        print("🔧  Implement batched_cosine.")
        return
    assert torch.allclose(out, _ref_batched_cosine(A, B), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 6 – One-hot
-------------------
Given an integer index tensor **idx** of shape `(B,)` and
num_classes =C, return a one-hot tensor `(B, C)` without any loop.

In [17]:
import torch

def one_hot(idx: torch.Tensor, num_classes: int) -> torch.Tensor:
    B, = idx.shape
    idx = idx.view(B, 1)
    val = torch.ones(B, 1)
    res = torch.zeros(B, num_classes)
    res.scatter_(1, idx, val)
    return res


def _ref_one_hot(idx, C):
    out = torch.zeros(idx.size(0), C, device=idx.device)
    out.scatter_(1, idx.unsqueeze(1), 1.0)
    return out

def _self_check():
    i = torch.tensor([2, 0, 1])
    try:
        out = one_hot(i, 4)
    except NotImplementedError:
        print("🔧  Implement one_hot.")
        return
    assert torch.equal(out, _ref_one_hot(i, 4))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 7 – Masked cumulative product
-------------------------------------
Input
* **X** ∈ ℝ^{B×T} (positive)
* **reset** ∈ {0,1}^{B×T}

For each row, compute the running product but **restart at 1** whenever
reset==1 at that timestep.

In [21]:
import torch

def masked_cumprod(X: torch.Tensor, reset: torch.Tensor) -> torch.Tensor:
    assert torch.all(X != 0.0), "assumption voilated"
    
    P = torch.cumprod(X, dim=1)
    PP = torch.cat((torch.ones_like(P[:, :1]), P[:, :-1]), dim=1)

    index = reset.cumsum(dim=1)
    tmp = torch.zeros_like(X)
    tmp.scatter_reduce_(1, index, reset * PP, reduce="sum")
    tmp[:, 0] = 1

    baseline = torch.gather(tmp, 1, index)
    return P / baseline

def _ref_masked_cumprod(X, reset):
    B, T = X.shape
    running = torch.ones(B)
    res = torch.ones(B, T)
    for t in range(T):
        running = torch.where(reset[:, t], X[:, t], running * X[:, t])
        res[:, t] = running
    return res
    
def _self_check():
    X = torch.tensor([[2.,3.,4.,5.],[1.,2.,3.,4.]])
    R = torch.tensor([[0,1,0,0],[0,0,1,0]], dtype=torch.bool)
    try:
        out = masked_cumprod(X, R)
    except NotImplementedError:
        print("🔧  Implement masked_cumprod.")
        return
    assert torch.allclose(out, _ref_masked_cumprod(X, R))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 8 – Conv2d (no bias) with unfold
----------------------------------------
Re-implement `torch.nn.functional.conv2d` for square kernels
using `torch.nn.functional.unfold`.

Signature
---------
conv2d_im2col(x, w, stride=1, padding=0)

In [1]:
import torch

def conv2d_im2col(x: torch.Tensor, w: torch.Tensor,
                  stride: int = 1, padding: int = 0) -> torch.Tensor:
    B, Ci, H, W = x.shape
    Co, _, k, _ = w.shape
    x_unf = torch.nn.functional.unfold(x, k, padding=padding, stride=stride) # B x (Ci * k * k) x L
    w = w.view(Co, Ci * k * k)
    out = torch.einsum("bdl,cd->bcl", x_unf, w)
    Ho = (H + 2*padding - k) // stride + 1
    Wo = (W + 2*padding - k) // stride + 1
    return out.view(B, Co, Ho, Wo)

def _ref_conv2d_im2col(x, w, stride=1, padding=0):
    N, C_in, H, W = x.shape
    C_out, _, k, _ = w.shape
    x_unf = torch.nn.functional.unfold(x, k, padding=padding, stride=stride)
    out = w.view(C_out, -1) @ x_unf
    H_out = (H + 2*padding - k)//stride + 1
    W_out = (W + 2*padding - k)//stride + 1
    return out.view(N, C_out, H_out, W_out)

def _self_check():
    x = torch.randn(1,3,5,5); w = torch.randn(4,3,3,3)
    try:
        out = conv2d_im2col(x, w, padding=1)
    except NotImplementedError:
        print("🔧  Implement conv2d_im2col.")
        return
    assert torch.allclose(out, _ref_conv2d_im2col(x, w, padding=1), atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 9 – Batched k-NN (squared Euclidean)
--------------------------------------------
Given X ∈ ℝ^{B×N×D}, Y ∈ ℝ^{B×M×D} and k, return indices of the k
closest Y-vectors for every X-vector within each batch.

In [5]:
import torch

def batched_knn(X: torch.Tensor, Y: torch.Tensor, k: int) -> torch.Tensor:
    B, N, D = X.shape
    B, M, D = Y.shape

    xx = (X * X).sum(dim=-1, keepdim=True)                  # B, N, 1
    yy = (Y * Y).sum(dim=-1, keepdim=True).permute(0, 2, 1) # B, 1, M
    xy = torch.einsum("bnd,bmd->bnm", X, Y)                 # B, N, M

    l2 = xx + yy - 2*xy                                     # B, N, M
    _, idx = l2.topk(k, dim=-1, largest=False)
    return idx


def _ref_batched_knn(X, Y, k):
    d = (X.pow(2).sum(-1, keepdim=True)
         + Y.pow(2).sum(-1).unsqueeze(1)
         - 2*torch.einsum("bnd,bmd->bnm", X, Y))
    _, idx = d.topk(k, dim=-1, largest=False)
    return idx

def _self_check():
    X,Y = torch.randn(2,4,6), torch.randn(2,5,6)
    try:
        idx = batched_knn(X, Y, 3)
    except NotImplementedError:
        print("🔧  Implement batched_knn.")
        return
    assert torch.equal(idx, _ref_batched_knn(X,Y,3))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 10 – Brevity penalty
----------------------------
For sentence lengths `len_c, len_r` (candidate vs reference),
return

    BP = 1                   if len_c > len_r  
         exp(1 − len_r/len_c) otherwise

Vectorized over a batch of shape `(B,)`.

In [6]:
import torch
import math

def brevity_penalty(len_c: torch.Tensor, len_r: torch.Tensor) -> torch.Tensor:
    len_c, len_r = len_c.float(), len_r.float()
    return torch.where(len_c > len_r, torch.ones_like(len_c), torch.exp(1 - len_r / (len_c + 1e-8)))


def _ref_bp(lc, lr):
    lc, lr = lc.float(), lr.float()
    return torch.where(lc > lr, torch.ones_like(lc), torch.exp(1 - lr / (lc + 1e-8)))

def _self_check():
    lc, lr = torch.tensor([10, 5]), torch.tensor([8, 9])
    try:
        bp = brevity_penalty(lc, lr)
    except NotImplementedError:
        print("🔧  Implement brevity_penalty.")
        return
    assert torch.allclose(bp, _ref_bp(lc, lr))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 11 – One-step GRU
-------------------------
The GRU update consists of the following steps:

1. Reset gate:
    r = sigmoid(Wx_r @ x + Wh_r @ h + b_r)

2. Update gate:
    z = sigmoid(Wx_z @ x + Wh_z @ h + b_z)

3. Candidate (new) state:
    n = tanh(Wx_n @ x + r ⊙ (Wh_n @ h) + b_n)

4. Final hidden state:
    h_next = (1 - z) ⊙ h + z ⊙ n

In [11]:
import torch

def gru_cell(x: torch.Tensor, h: torch.Tensor,
             Wx: torch.Tensor, Wh: torch.Tensor,
             b: torch.Tensor) -> torch.Tensor:
    #  x:  B, D
    #  h:  B, H
    # Wx: 3H, D
    # Wh: 3H, H
    #  b: 3H
    B, D = x.shape
    _, H = h.shape

    x_proj = x @ Wx.T
    h_proj = h @ Wh.T
    b_proj = b

    x_r, x_z, x_n = x_proj.chunk(3, dim=-1)
    h_r, h_z, h_n = h_proj.chunk(3, dim=-1)
    b_r, b_z, b_n = b_proj.chunk(3, dim=-1)

    r = torch.sigmoid(x_r + h_r + b_r)
    z = torch.sigmoid(x_z + h_z + b_z)
    n = torch.tanh(x_n + r * h_n + b_n)

    return (1 - z) * h + z * n
    


def _ref_gru_cell(x,h,Wx,Wh,b):
    H = h.size(1)

    # Input and hidden projections for all 3 gates
    x_proj = x @ Wx.T + b      # (B, 3H)
    h_proj = h @ Wh.T          # (B, 3H)

    # Split into reset, update, and new gate
    x_r, x_z, x_n = x_proj.chunk(3, dim=-1)
    h_r, h_z, h_n = h_proj.chunk(3, dim=-1)

    # Gates
    r = torch.sigmoid(x_r + h_r)             # Reset gate
    z = torch.sigmoid(x_z + h_z)             # Update gate
    n = torch.tanh(x_n + r * h_n)            # New gate with r ⊙ h_n

    # Final hidden state
    h_next = (1 - z) * h + z * n             # GRU interpolation
    return h_next

def _self_check():
    B,D,H=2,3,4
    args=[torch.randn(B,D),torch.randn(B,H),
          torch.randn(3*H,D),torch.randn(3*H,H),
          torch.randn(3*H)]
    try:
        out=gru_cell(*args)
    except NotImplementedError:
        print("🔧  Implement gru_cell.")
        return
    assert torch.allclose(out,_ref_gru_cell(*args),atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 12 – Causal self-attention scores
-----------------------------------------
Given Q,K ∈ ℝ^{B×T×d}, return the softmaxed score matrix
`(B, T, T)` with an upper-triangular causal mask.

In [20]:
import torch, math

def causal_attention_scores(Q: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    # Q: B, T, D
    # K: B, S, D
    B, T, D = Q.shape
    _, S, _ = K.shape
    
    mask = torch.tril(torch.ones(T, S, dtype=torch.bool), diagonal=0)
    
    dot_products = torch.einsum("btd,bsd->bts", Q, K) / math.sqrt(D)
    dot_products.masked_fill_(~mask, -torch.inf)

    attn_weights = torch.softmax(dot_products, dim=-1)
    return attn_weights
    

def _ref_causal(Q,K):
    d = Q.size(-1)
    S = torch.einsum("btd,bkd->btk",Q,K)/math.sqrt(d)
    T = S.size(-1)

    mask = torch.triu(torch.ones(T,T,device=Q.device,dtype=torch.bool),1)
    return torch.softmax(S.masked_fill(mask,float("-inf")), -1)

def _self_check():
    Q,K=torch.randn(1,6,4),torch.randn(1,6,4)
    try: out=causal_attention_scores(Q,K)
    except NotImplementedError:
        print("🔧  Implement causal_attention_scores.")
        return
    assert torch.allclose(out,_ref_causal(Q,K),atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 13 – Min–max scale every row
------------------------------------
For X ∈ ℝ^{B×D} return

    Yᵢⱼ = (Xᵢⱼ − min(Xᵢ)) / (max(Xᵢ) − min(Xᵢ) + ε)

so every row is mapped to the range [0, 1].

No Python loops allowed.

In [22]:
import torch

def minmax_rows(X: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    # X: (B, D)
    Xmin, _ = torch.min(X, dim=-1, keepdim=True)
    Xmax, _ = torch.max(X, dim=-1, keepdim=True)

    Y = (X - Xmin) / (Xmax - Xmin + eps)
    return Y


# reference & test
def _ref_minmax_rows(X, eps=1e-8):
    xmin = X.min(dim=1, keepdim=True).values
    xmax = X.max(dim=1, keepdim=True).values
    return (X - xmin) / (xmax - xmin + eps)

def _self_check():
    X = torch.randn(4, 7)
    try:
        out = minmax_rows(X)
    except NotImplementedError:
        print("🔧  Implement minmax_rows.")
        return
    assert torch.allclose(out, _ref_minmax_rows(X), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 14 – Batch RGB→gray conversion
--------------------------------------
Given imgs ∈ ℝ^{N×3×H×W} with values in [0,1], return a grayscale
tensor of shape (N, 1, H, W) using weights

    Y = 0.299 R + 0.587 G + 0.114 B

In [23]:
import torch

def rgb_to_gray(imgs: torch.Tensor) -> torch.Tensor:
    # imgs: B, C, H, W
    w = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
    return (imgs * w).sum(dim=1, keepdim=True)


def _ref_rgb_to_gray(imgs):
    w = torch.tensor([0.299, 0.587, 0.114], device=imgs.device).view(1,3,1,1)
    return (imgs * w).sum(1, keepdim=True)

def _self_check():
    imgs = torch.rand(2,3,8,8)
    try:
        out = rgb_to_gray(imgs)
    except NotImplementedError:
        print("🔧  Implement rgb_to_gray.")
        return
    assert torch.allclose(out, _ref_rgb_to_gray(imgs), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 15 – σ(z) and σ'(z)
---------------------------
Return both the logistic sigmoid and its derivative for an
arbitrary-shaped tensor z (no loops).

In [24]:
import torch

def sigmoid_and_grad(z: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    s = torch.sigmoid(z)
    return s, s*(1-s)

def _ref_sig_and_grad(z):
    s = torch.sigmoid(z)
    return s, s*(1-s)

def _self_check():
    z = torch.randn(5,3,2)
    try:
        s, ds = sigmoid_and_grad(z)
    except NotImplementedError:
        print("🔧  Implement sigmoid_and_grad.")
        return
    s_ref, ds_ref = _ref_sig_and_grad(z)
    assert torch.allclose(s, s_ref) and torch.allclose(ds, ds_ref)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 16 – Trace & diagonal per batch
---------------------------------------
For M ∈ ℝ^{B×D×D} return

    trace  – shape (B,)
    diag   – shape (B,D)

using only tensor ops.

In [25]:
import torch

def batch_trace_diag(M: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    # M: (B, D, D)
    diag = M.diagonal(dim1=1, dim2=2)
    trace = diag.sum(dim=-1)
    return trace, diag


def _ref_batch_trace_diag(M):
    diag = M.diagonal(dim1=-2, dim2=-1)
    return diag.sum(-1), diag

def _self_check():
    M = torch.randn(3,4,4)
    try:
        tr, d = batch_trace_diag(M)
    except NotImplementedError:
        print("🔧  Implement batch_trace_diag.")
        return
    tr_ref, d_ref = _ref_batch_trace_diag(M)
    assert torch.allclose(tr, tr_ref) and torch.allclose(d, d_ref)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 17 – Label-smoothed CE loss
-----------------------------------
Given logits (B,C), integer targets (B,) and smoothing α ∈ [0,1),
return the scalar loss

    L = − mean_i  sum_c  q_ic  log softmax_i,c

where q is the smoothed one-hot:  
 q_i,target = 1−α, others = α/(C−1).

No loops.

In [26]:
import torch, torch.nn.functional as F

def ls_ce_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float) -> torch.Tensor:
    B, C = logits.shape
    q = torch.ones_like(logits) * (alpha / (C-1)) # B, C
    q.scatter_(1, targets.view(B, 1), 1-alpha)    # B, C
    return -(q * torch.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

def _ref_ls_ce(logits, targets, alpha):
    B,C = logits.shape
    q = torch.full((B,C), alpha/(C-1), device=logits.device)
    q.scatter_(1, targets.unsqueeze(1), 1-alpha)
    return -(q * F.log_softmax(logits,1)).sum(1).mean()

def _self_check():
    logit = torch.randn(6,10)
    tgt   = torch.randint(0,10,(6,))
    try:
        loss = ls_ce_loss(logit, tgt, 0.1)
    except NotImplementedError:
        print("🔧  Implement ls_ce_loss.")
        return
    assert torch.allclose(loss, _ref_ls_ce(logit,tgt,0.1), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 18 – Jaccard indices
----------------------------
A, B ∈ {0,1}^{B×N×D}.  
Compute J[b, i] = |A[b,0] ∩ B[b,i]| / |A[b,0] ∪ B[b,i]|.

Return J shape (B, N).

In [27]:
import torch

def jaccard_pair(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    A0 = A[:, :1, :] # B, 1, D
    inter = (A0 & B).sum(-1)
    union = (A0 | B).sum(-1)
    return inter / (union + 1e-8)


def _ref_jaccard_pair(A,B):
    inter = (A[:,:1] & B).sum(-1)
    union = (A[:,:1] | B).sum(-1)
    return inter / (union + 1e-8)

def _self_check():
    A = (torch.rand(2,1,5)>.5).bool().repeat(1,3,1)
    B = (torch.rand(2,3,5)>.5).bool()
    try:
        out = jaccard_pair(A,B)
    except NotImplementedError:
        print("🔧  Implement jaccard_pair.")
        return
    assert torch.allclose(out, _ref_jaccard_pair(A,B), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 19 – Dice coefficient (mean over classes)
-------------------------------------------------
pred, mask ∈ {0,1}^{N×C×H×W}.  
Return the mean Dice over the C classes:

    Dice_c = (2|∩| + ε) / (|pred_c| + |mask_c| + ε)

In [28]:
import torch

def dice_coeff(pred: torch.Tensor, mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    inter = (pred & mask).sum((0,2,3))
    union = pred.sum((0,2,3)) + mask.sum((0,2,3))
    return ((2*inter+eps)/(union+eps)).mean()


def _ref_dice(pred, mask, eps=1e-6):
    inter = (pred & mask).sum((0,2,3))
    union = pred.sum((0,2,3)) + mask.sum((0,2,3))
    return ((2*inter+eps)/(union+eps)).mean()

def _self_check():
    p = (torch.rand(4,3,10,10)>.5)
    m = (torch.rand(4,3,10,10)>.5)
    try:
        d = dice_coeff(p,m)
    except NotImplementedError:
        print("🔧  Implement dice_coeff.")
        return
    assert torch.allclose(d, _ref_dice(p,m), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 20 – Causal conv1d (no bias)
------------------------------------
x: (B,C,T), w: (C_out,C,k).  
Pad (k−1) zeros on the *left* so output has length T.
No loops; use `unfold` or `tensor.unfold`.

In [29]:
import torch

def causal_conv1d(x: torch.Tensor, w: torch.Tensor, dilation: int = 1) -> torch.Tensor:
    B, C_in, T = x.shape
    C_out, C_in, k = w.shape

    x_pad = torch.nn.functional.pad(x, (k-1,0), "constant", 0)
    x_unf = x_pad.unfold(2,k,1)                                 # B, C_in, T, k
    return torch.einsum("bctk,ock->bot", x_unf, w)

def _ref_causal_conv1d(x,w,d=1):
    k = w.size(-1)
    pad = (k-1)*d
    xpad = torch.nn.functional.pad(x,(pad,0))
    x_unf = xpad.unfold(2,k,1).transpose(1,2)  # (B,T,C,k)
    return torch.einsum("btck,ock->bot", x_unf, w)

def _self_check():
    x = torch.randn(1,2,6)
    w = torch.randn(3,2,3)
    try:
        out = causal_conv1d(x,w)
    except NotImplementedError:
        print("🔧  Implement causal_conv1d.")
        return
    assert torch.allclose(out, _ref_causal_conv1d(x,w), atol=1e-5)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 21 – Rolling mean
-------------------------
For x ∈ ℝ^{B×T} and window w (w≤T) return
shape (B, T−w+1) where each entry is the mean of a sliding length-w
window.  Use tensor.unfold; no loops.

In [31]:
import torch

def rolling_mean(x: torch.Tensor, window: int) -> torch.Tensor:
    B, T = x.shape
    x_unf = x.unfold(1,window,1).mean(-1)
    return x_unf

def _ref_rolling_mean(x, w):
    return x.unfold(1,w,1).mean(-1)

def _self_check():
    a = torch.arange(6).float().view(1,-1)
    try:
        out = rolling_mean(a,3)
    except NotImplementedError:
        print("🔧  Implement rolling_mean.")
        return
    assert torch.allclose(out, _ref_rolling_mean(a,3))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 22 – Gather (B, T, D) at time indices
---------------------------------------------
seq: (B,T,D), idx: (B,K) → out: (B,K,D) using `tensor.gather`.

In [32]:
import torch

def batch_gather(seq: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    B, T, D = seq.shape
    B, K = idx.shape
    idx = idx.view(B, K, 1).expand(B, K, D)
    return torch.gather(seq, 1, idx)


def _ref_batch_gather(seq, idx):
    return seq.gather(1, idx.unsqueeze(-1).expand(-1,-1,seq.size(-1)))

def _self_check():
    s = torch.randn(2,5,4)
    i = torch.tensor([[1,3],[0,4]])
    try:
        out = batch_gather(s,i)
    except NotImplementedError:
        print("🔧  Implement batch_gather.")
        return
    assert torch.allclose(out, _ref_batch_gather(s,i))
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 23 – Sparsemax
----------------------
Implement sparsemax (Martins & Astudillo 2016) along a given
dimension dim (default −1).  See paper for algorithm.

In [9]:
import torch

def sparsemax(z: torch.Tensor, dim: int = -1) -> torch.Tensor:
    z_sorted, _ = torch.sort(z, dim=dim, descending=True)
    z_cumsum = z_sorted.cumsum(dim)
    
    k = torch.arange(1, z.size(dim)+1)
    k_shape = [1] * z.dim()
    k_shape[dim] = -1
    k = k.view(*k_shape)
    
    support = (z_cumsum - k * z_sorted) < 1
        
    tau = ((z_sorted * support).sum(dim=dim, keepdim=True) - 1) / support.sum(dim=dim, keepdim=True)

    return torch.clamp(z - tau, min=0)
    

def _ref_sparsemax(z, dim=-1):
    # 1) sort z in descending order
    z_sorted, _ = torch.sort(z, dim=dim, descending=True)

    # 2) compute running sums of sorted z, subtract 1
    z_cumsum = z_sorted.cumsum(dim) - 1

    # 3) create the 1-based index k for each position along `dim`
    r = torch.arange(1, z.size(dim) + 1, device=z.device, dtype=z.dtype)
    # reshape so it can broadcast along every other dimension
    r_shape = [1] * z.dim()
    r_shape[dim] = -1
    r = r.view(*r_shape)

    # 4) find the support: z_k > tau_k
    support = z_sorted > z_cumsum / r
    k_support = support.sum(dim=dim, keepdim=True)

    # 5) compute the threshold τ
    tau = ( (z_sorted * support).sum(dim=dim, keepdim=True) - 1 ) / k_support

    # 6) project onto the simplex
    return torch.clamp(z - tau, min=0.0)

def _self_check():
    z = torch.randn(3,7)
    try:
        out = sparsemax(z)
    except NotImplementedError:
        print("🔧  Implement sparsemax.")
        return
    assert torch.allclose(out, _ref_sparsemax(z), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 24 – Multi-head causal self-attention (no bias)
------------------------------------------------------
Q,K,V ∈ ℝ^{B×T×H×d}.  
Return context: (B,T,H,d) using dot-product attention with an
upper-triangular causal mask (each position attends to ≤ its past).
No loops.

In [12]:
import torch, math

def mh_causal_attn(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
    # Q: B, T, H, D
    # K: B, S, H, D
    # V: B, S, H, D

    B, T, H, D = Q.shape

    scores = torch.einsum("bthd,bshd->bhts", Q, K) / math.sqrt(D)
    masks = torch.tril(torch.ones(T, T, dtype=torch.bool), diagonal=0).view(1, 1, T, T)
    scores.masked_fill_(~masks, -torch.inf)
    weights = torch.softmax(scores, dim=-1)
    return torch.einsum("bhts,bshd->bthd", weights, V)
    

def _ref_mh_attn(Q,K,V):
    d = Q.size(-1)
    scores = torch.einsum("bthd,bshd->bhts", Q, K)/math.sqrt(d)
    T = scores.size(-1)
    mask = torch.triu(torch.ones(T,T,device=Q.device,dtype=torch.bool),1)
    scores = scores.masked_fill(mask, float("-inf"))
    A = torch.softmax(scores, -1)
    return torch.einsum("bhts,bshd->bthd", A, V)

def _self_check():
    Q = torch.randn(2,6,4,8)
    K = torch.randn(2,6,4,8)
    V = torch.randn(2,6,4,8)
    try:
        out = mh_causal_attn(Q,K,V)
    except NotImplementedError:
        print("🔧  Implement mh_causal_attn.")
        return
    assert torch.allclose(out, _ref_mh_attn(Q,K,V), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 25 – Shannon entropy
----------------------------
Given a batch of **probability distributions** `p` whose entries
sum to 1 along a chosen dimension `dim` (default = −1), compute

        H(p) = − Σᵢ pᵢ log pᵢ          (natural log)

Return a tensor with one fewer dimension than the input, the
summation being performed over `dim`.

* Use **no Python for-loops**.
* Add a small ε before log to avoid `log(0)`.

In [3]:
import torch

def entropy(p: torch.Tensor, dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    """Implement me!"""
    assert torch.all(p >= 0) and torch.allclose(p.sum(dim=dim), torch.tensor(1.0)), "invalid input"
    return (-p * torch.log(p + eps)).sum(dim=dim)
    
# ---------- reference + tests ----------
def _ref_entropy(p, dim=-1, eps=1e-8):
    return -(p * (p + eps).log()).sum(dim)

def _self_check():
    p = torch.tensor([[0.5, 0.5],
                      [0.25, 0.75]])
    try:
        H = entropy(p)
    except NotImplementedError:
        print("🔧  Implement entropy and re-run the cell.")
        return
    assert torch.allclose(H, _ref_entropy(p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 26 – Cross-entropy
--------------------------
Given two *matching* probability tensors **q** (the “true” distribution)
and **p** (the “predicted” distribution) of shape `(..., C)`, compute

        CE(q, p) = − Σᵢ qᵢ log pᵢ

along `dim` (default = −1).  Return a tensor of shape equal to
`q` with that dimension removed.

Assume `q` already sums to 1 (it can be soft or one-hot).
No loops; add ε for log-safety.

In [4]:
import torch

def cross_entropy(q: torch.Tensor, p: torch.Tensor,
                  dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    return (-q * torch.log(p + eps)).sum(dim=dim)


def _ref_cross_entropy(q, p, dim=-1, eps=1e-8):
    return -(q * (p + eps).log()).sum(dim)

def _self_check():
    q = torch.tensor([[0., 1., 0.],
                      [0.2, 0.3, 0.5]])
    p = torch.tensor([[0.1, 0.8, 0.1],
                      [0.3, 0.2, 0.5]])
    try:
        ce = cross_entropy(q, p)
    except NotImplementedError:
        print("🔧  Implement cross_entropy.")
        return
    assert torch.allclose(ce, _ref_cross_entropy(q, p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 27 – KL divergence  D_KL(q || p)
----------------------------------------
For two probability tensors **q** (reference) and **p** (model)
of identical shape `(..., C)`, compute

        KL(q‖p) = Σᵢ qᵢ log(qᵢ / pᵢ)

over `dim` (default = −1).  The output’s shape lacks that dimension.

Requirements
------------
* Pure tensor ops, no loops.
* Add ε to both q and p inside the log to avoid division by zero.

In [5]:
import torch

def kl_divergence(q: torch.Tensor, p: torch.Tensor,
                  dim: int = -1, eps: float = 1e-8) -> torch.Tensor:
    return (q * (torch.log(q + eps) - torch.log(p + eps))).sum(dim=dim)


def _ref_kl(q, p, dim=-1, eps=1e-8):
    q_safe = q + eps
    p_safe = p + eps
    return (q_safe * (q_safe / p_safe).log()).sum(dim)

def _self_check():
    q = torch.tensor([[0.6, 0.4]])
    p = torch.tensor([[0.5, 0.5]])
    try:
        kl = kl_divergence(q, p)
    except NotImplementedError:
        print("🔧  Implement kl_divergence.")
        return
    assert torch.allclose(kl, _ref_kl(q, p), atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 28 – Sinusoidal positional encoding
-------------------------------------------
Implement the original Transformer positional embedding

    PE[pos, 2i]     =  sin( pos / 10000^(2i/d) )
    PE[pos, 2i + 1] =  cos( pos / 10000^(2i/d) )

for sequence length `seq_len` and hidden size `d` (d must be even).

Return a tensor of shape **(seq_len, d)** on the requested
`device` / `dtype` – and **use no Python loops**.

In [7]:
import torch
import math
from typing import Optional

def compute_sin_cos(T, D):
    # sin(t * 10000^(-i/(D/2)))
    # cos(t * 10000^(-i/(D/2)))

    H = D//2
    tt = torch.arange(0, T).view(T, 1)
    ii = torch.arange(0, H).view(1, H)
    ff = 10000**(-ii/H)

    sin = torch.sin(tt * ff)
    cos = torch.cos(tt * ff)

    return sin, cos
    
def sinusoidal_positional_encoding(seq_len: int,
                                   d: int,
                                   device: Optional[torch.device] = None,
                                   dtype: torch.dtype = torch.float32
                                   ) -> torch.Tensor:
    """Implement me!"""
    assert d % 2 == 0, "d must be even"
    sin, cos = compute_sin_cos(seq_len, d)
    pe = torch.stack((sin, cos), dim=-1).reshape(seq_len, d)
    return pe


# -----------------------------------------------------------
def _ref_sinusoidal(seq_len, d, device=None, dtype=torch.float32):
    if d % 2:
        raise ValueError("d must be even.")
    pos = torch.arange(seq_len, device=device).unsqueeze(1)          # (T,1)
    i   = torch.arange(0, d, 2, device=device).unsqueeze(0)          # (1,d/2)
    angle = pos / torch.pow(10000.0, i / d)                          # (T,d/2)
    pe = torch.zeros(seq_len, d, device=device, dtype=dtype)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe

def _self_check():
    T, D = 5, 8
    try:
        pe = sinusoidal_positional_encoding(T, D)
    except NotImplementedError:
        print("🔧  Implement sinusoidal_positional_encoding.")
        return
    assert pe.shape == (T, D), "Shape mismatch"
    assert torch.allclose(pe, _ref_sinusoidal(T, D), atol=1e-6), "Wrong values"
    # Sanity: first position should be [0,1,0,1,...]
    first = pe[0]
    assert torch.allclose(first[0::2], torch.zeros(D//2))            # sin(0)=0
    assert torch.allclose(first[1::2], torch.ones (D//2))            # cos(0)=1
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 29 – Rotary positional embedding (RoPE)
-----------------------------------------------
Apply rotary position encodings (Su et al., 2021) to **query** and
**key** tensors of shape (B, T, d) where d is even.

Algorithm (vector form)
~~~~~~~~~~~~~~~~~~~~~~~
Split last dim into even/odd pairs:

    x_even, x_odd = x[..., 0::2], x[..., 1::2]

Define frequency vector:
    θ_i = 10000^{ -2i / d }   for i = 0 … d/2−1

For each position **pos** (0-based):

    sin  = sin(pos * θ)
    cos  = cos(pos * θ)

Rotate each pair:

    x_rot_even =  x_even * cos - x_odd * sin
    x_rot_odd  =  x_even * sin + x_odd * cos

Interleave the two parts back together to shape (B,T,d).

Return **(q_rot, k_rot)**.

Requirements
------------
* Pure tensor ops, **no explicit Python loops**.

In [10]:
import torch
import math
from typing import Tuple

def compute_sin_cos(T, D):
    # sin(t * 10000^(-i/(D/2)))
    # cos(t * 10000^(-i/(D/2)))

    H = D//2
    tt = torch.arange(0, T).view(T, 1)
    ii = torch.arange(0, H).view(1, H)
    ff = 10000**(-ii/H)

    sin = torch.sin(tt * ff)
    cos = torch.cos(tt * ff)

    return sin, cos

def apply_rope(q: torch.Tensor,
               k: torch.Tensor
               ) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Args
    ----
    q, k : (B, T, d)  with even d

    Returns
    -------
    q_rot, k_rot  – same shapes after rotary encoding.
    """
    B, T, D = q.shape
    sin, cos = compute_sin_cos(T, D)

    def rope(x):
        x0 = x[:, :, 0::2]
        x1 = x[:, :, 1::2]

        y0 = x0 * cos - x1 * sin
        y1 = x0 * sin + x1 * cos

        y = torch.stack((y0, y1), dim=-1).reshape(B, T, D)

        return y

    return rope(q), rope(k)

# -----------------------------------------------------------
def _rope_rotate(x):
    B,T,d = x.shape
    half  = d // 2
    freqs = torch.pow(10000.0, -torch.arange(0, half, device=x.device) / half)
    pos   = torch.arange(T, device=x.device).float()                # (T,)
    sin   = torch.sin( pos.unsqueeze(1) * freqs )                   # (T,half)
    cos   = torch.cos( pos.unsqueeze(1) * freqs )
    sin, cos = sin.unsqueeze(0), cos.unsqueeze(0)                   # (1,T,half)
    x_even, x_odd = x[..., 0::2], x[..., 1::2]                      # (B,T,half)
    rot_even = x_even * cos - x_odd * sin
    rot_odd  = x_even * sin + x_odd * cos
    # interleave
    x_rot = torch.empty_like(x)
    x_rot[..., 0::2] = rot_even
    x_rot[..., 1::2] = rot_odd
    return x_rot

def _ref_apply_rope(q, k):
    return _rope_rotate(q), _rope_rotate(k)

def _self_check():
    q = torch.randn(2, 7, 12)
    k = torch.randn(2, 7, 12)
    try:
        q_r, k_r = apply_rope(q, k)
    except NotImplementedError:
        print("🔧  Implement apply_rope.")
        return
    q_ref, k_ref = _ref_apply_rope(q, k)
    assert torch.allclose(q_r, q_ref, atol=1e-6)
    assert torch.allclose(k_r, k_ref, atol=1e-6)
    print("✅  basic tests passed")

_self_check()

✅  basic tests passed


Problem 30 – Connectionist-Temporal Classification (CTC) Loss
------------------------------------------------------------
Write CTC loss **from scratch** (forward + backward) using PyTorch’s
low-level `torch.autograd.Function`.

•  **Signature** must match `torch.nn.CTCLoss`:
       loss = CustomCTCLoss(blank=0, reduction="mean", zero_infinity=False)
       loss(log_probs, targets, input_lengths, target_lengths)

•  **log_probs**  (T, N, C) — log-softmax probabilities  
   **targets**    (N, S)   — padded with a special value (default 0)  
   **input_lengths**  (N,) — actual lengths ≤ T  
   **target_lengths** (N,) — ≤ S

•  Implement the *exclusive* β-variable version of the backward dynamic
   program (β_t(s) excludes the current label at time t).

•  No need to handle GPU / AMP edge cases; loops are fine for the small
  unit tests below, but try to keep your code vectorised where you can.

-----------------------------------------------------------------
Skeleton
~~~~~~~~
1.  **forward**
      – compute CTC loss for each batch element  
      – save anything you need on ctx for backward  
      – return either mean or sum according to `reduction`

2.  **backward**
      – input grad is `grad_out * dLoss/dLogProbs` (shape = log_probs)  
      – return gradients in the order of forward’s inputs  
      – all grads for integer tensors must be **None**

Tips
~~~~
* Expand each target with blanks: e.g. target=[g,a,t] → [blk,g,blk,a,blk,t,blk].
* α/β recurrences (log-space) are stable:
      α_t(s) = logsumexp( α_{t-1}(s)        + lp_t(s),
                           α_{t-1}(s-1)      + lp_t(s),
                           α_{t-1}(s-2) (if label changes) + lp_t(s) )
      β_t(s) = logsumexp( β_{t+1}(s)        + lp_{t+1}(s),
                           β_{t+1}(s+1)      + lp_{t+1}(s+1),
                           β_{t+1}(s+2) (if label changes) + lp_{t+1}(s+2) )
  with **exclusive** β meaning β_t does *not* include lp_t.
* The gradient wrt log_probs is  exp(α+β-logZ), reduced over s.

Feel free to peek at `torch.nn.functional.ctc_loss` for reference, but
do **not** call it inside your implementation.

------------------------------------------------------------
Starter code below — fill in the two `raise NotImplementedError` blocks.

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

NUM_INF = torch.tensor(1e20)

def extend_target(target, blank=0):
    L = len(target)
    out = torch.full((2*L+1,), blank)
    out[1::2] = target
    return out

def logsumexp(a, b):
    a, b = torch.maximum(a, b), torch.minimum(a, b)
    return a + torch.log1p(torch.exp(b-a))

# -------------------  YOUR IMPLEMENTATION AREA  -------------------
class CustomCTCLossFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                log_probs: torch.Tensor,        # (maxT,N,C)  log-softmax already
                targets: torch.Tensor,          # (N,maxL)
                input_lengths: torch.Tensor,    # (N,)
                target_lengths: torch.Tensor,   # (N,)
                blank: int,
                reduction: str,
                zero_infinity: bool,
                gradient_version: str) -> torch.Tensor:
        """
        Returns a scalar (if reduction != 'none') or shape (N,) tensor.
        Save everything you need on ctx for backward.
        """
        maxT, N, C = log_probs.shape
        N, maxL = targets.shape

        losses = []
        grads = torch.zeros_like(log_probs)
        for n in range(N):
            T = input_lengths[n]
            L = target_lengths[n]
            S = 2*L + 1

            lgp = log_probs[:T, n, :]             # T, C
            tgt = targets[n, :L]                  # L
            ext = extend_target(tgt, blank=blank) # 2L+1

            # compute alpha
            alpha = torch.full((T, S), -NUM_INF, dtype=log_probs.dtype)
            alpha[0, 0] = lgp[0, ext[0]]
            alpha[0, 1] = lgp[0, ext[1]]
            #
            for t in range(1, T):
                for s in range(S):
                    # frame t, emission ext[s]
                    stay = alpha[t-1, s-0] + lgp[t, ext[s]]
                    add1 = alpha[t-1, s-1] + lgp[t, ext[s]] if (s-1 >= 0) else -NUM_INF
                    add2 = alpha[t-1, s-2] + lgp[t, ext[s]] if (s-2 >= 0 and ext[s-1] == blank and ext[s-2] != ext[s]) else -NUM_INF

                    alpha[t, s] = logsumexp(logsumexp(stay, add1), add2)

            logZa = logsumexp(alpha[T-1, S-2], alpha[T-1, S-1])

            # compute beta
            beta = torch.full((T, S), -NUM_INF, dtype=log_probs.dtype)
            beta[T-1, S-2] = 0
            beta[T-1, S-1] = 0
            #
            for t in reversed(range(T-1)):
                for s in reversed(range(S)):
                    # frame t, multiple emission possibility
                    stay = lgp[t+1, ext[s+0]] + beta[t+1, s+0]
                    add1 = lgp[t+1, ext[s+1]] + beta[t+1, s+1] if (s+1 < S) else -NUM_INF
                    add2 = lgp[t+1, ext[s+2]] + beta[t+1, s+2] if (s+2 < S and ext[s+1] == blank and ext[s+2] != ext[s]) else -NUM_INF

                    beta[t, s] = logsumexp(logsumexp(stay, add1), add2)

            logZb = logsumexp(lgp[0, ext[0]] + beta[0, 0], lgp[0, ext[1]] + beta[0, 1])

            # compute loss
            assert torch.allclose(logZa, logZb), f"mismatch from alpha and beta computation, {logZa}, {logZb}"
            logZ = (logZa + logZb) / 2
            losses.append(-logZ)

            # compute gamma
            gamma = alpha + beta - logZ

            for s in range(S):
                grads[:T, n, ext[s]] -= torch.exp(gamma[:, s])

            if gradient_version == "torch":
                grads[:T, n, :] += torch.exp(lgp[:T, :])

        losses = torch.tensor(losses)

        ctx.save_for_backward(grads)

        return losses

    
    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        grad_out is either shape (N,) or scalar depending on reduction.
        Must return gradients for the *same* number of inputs received
        in forward:  log_probs, targets, input_lengths, target_lengths,
        blank, reduction, zero_infinity.  Gradients for non-tensor or
        int/str args should be None.
        """

        N, = grad_out.shape
        grads, = ctx.saved_tensors
        grad_in = grad_out.reshape(1, N, 1) * grads
        
        return grad_in, None, None, None, None, None, None, None


class CustomCTCLoss(nn.Module):
    """
    Thin wrapper to match nn.CTCLoss exact call signature.
    """
    def __init__(self,
                 blank: int = 0,
                 reduction: str = "mean",
                 zero_infinity: bool = False,
                 gradient_version: str = "true"):
        super().__init__()
        if reduction not in ("none", "mean", "sum"):
            raise ValueError("reduction must be 'none', 'mean' or 'sum'")
        if gradient_version not in ("true", "torch"):
            raise ValueError("gradient_version must be 'true' or 'torch'")
        self.blank = blank
        self.reduction = reduction
        self.zero_infinity = zero_infinity
        self.gradient_version = gradient_version

    def forward(self,
                log_probs: torch.Tensor,
                targets: torch.Tensor,
                input_lengths: torch.Tensor,
                target_lengths: torch.Tensor) -> torch.Tensor:
        losses = CustomCTCLossFunction.apply(
            log_probs, targets, input_lengths, target_lengths,
            self.blank, self.reduction, self.zero_infinity, self.gradient_version
        )
        if self.reduction == "none":
            loss = losses
        elif self.reduction == "mean":
            loss = torch.mean(losses / target_lengths)
        elif self.reduction == "sum":
            loss = torch.sum(losses)
        return loss
# ------------------------------------------------------------------


# -------------------  REFERENCE + SELF-CHECK  ---------------------
def _ref_ctc_loss(log_probs, targets, input_lengths, target_lengths,
                  blank=0, reduction="mean", zero_infinity=False):
    return F.ctc_loss(log_probs, targets, input_lengths, target_lengths,
                      blank=blank, reduction=reduction,
                      zero_infinity=zero_infinity)

def _self_check():
    torch.manual_seed(0)
    T, N, C = 20, 3, 4
    S = 9
    log_probs = F.log_softmax(torch.randn(T, N, C), dim=-1).requires_grad_()
    targets = torch.randint(1, C, (N, S), dtype=torch.long)
    input_lengths  = torch.full((N,), T, dtype=torch.long)
    target_lengths = torch.randint(1, S+1, (N,), dtype=torch.long)

    my_ctc  = CustomCTCLoss(blank=0, reduction="mean", gradient_version="torch")
    ref_ctc = nn.CTCLoss(blank=0, reduction="mean")

    try:
        loss_my  = my_ctc(log_probs, targets, input_lengths, target_lengths)
    except NotImplementedError:
        print("🔧  Implement CustomCTCLossFunction.forward/backward and re-run.")
        return

    loss_ref = ref_ctc(log_probs, targets, input_lengths, target_lengths)

    # Forward value check
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-4), \
        f"Loss mismatch: {loss_my} vs {loss_ref}"

    # Gradient check (wrt log_probs only)
    loss_ref.backward()
    grad_ref = log_probs.grad.clone()
    log_probs.grad.zero_()
    loss_my.backward()
    grad_my = log_probs.grad

    assert torch.allclose(grad_my, grad_ref, atol=1e-4), \
        "Gradient mismatch w.r.t. log_probs"

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


### Note PyTorch implementation of the CTC gradient is incorrect.

In [45]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
T, N, C = 20, 3, 4
S = 9

log_probs = F.log_softmax(torch.randn(T, N, C, dtype=torch.float64), dim=2).requires_grad_()
targets = torch.randint(1, C, (N, S), dtype=torch.long)
input_lengths = torch.full((N,), T, dtype=torch.long)
target_lengths = torch.randint(1, S + 1, (N,), dtype=torch.long)

custom_ctc_loss = CustomCTCLoss(blank=0, reduction="mean", gradient_version="true")

torch.autograd.gradcheck(custom_ctc_loss, (logits, targets, input_lengths, target_lengths), eps=1e-6, atol=1e-4)

True

Problem 31 – InfoNCE loss used in CLIP
-------------------------------------
Given L2-normalised **image** and **text** feature matrices

    images : (B, D)
    texts  : (B, D)

compute the symmetric InfoNCE loss

        L = ½ · [ CE( logits_img2txt, targets ) +
                  CE( logits_txt2img, targets ) ]

where

    logits_img2txt =  (images · textsᵀ) / τ      # (B, B)
    logits_txt2img =  logits_img2txtᵀ
    targets = torch.arange(B)                    # 0 … B−1

`τ` is a positive **temperature** scalar (default 0.07).

Return a scalar loss.  Use **no explicit Python loops**.

**Your task:**
Implement the function `info_nce_clip` below.

Afterwards, the self-check harness will compare your value and
gradient against a reference implementation.
"""

In [6]:
import torch
import torch.nn.functional as F
from typing import Optional

# ------------------------ YOUR CODE HERE ------------------------
def info_nce_clip(image_feats: torch.Tensor,
                  text_feats: torch.Tensor,
                  temperature: float = 0.07) -> torch.Tensor:
    """
    Args
    ----
    image_feats : (B, D)  *L2-normalised*
    text_feats  : (B, D)  *L2-normalised*
    temperature : positive scalar τ

    Returns
    -------
    Scalar InfoNCE loss (float tensor, requires_grad=True if inputs do).
    """
    # image_feats: B, D
    # text_feats: C, D
    B, D = image_feats.shape
    logits_i2t = torch.einsum("bd,cd->bc", image_feats, text_feats) / temperature
    logits_t2i = logits_i2t.permute(1, 0)
    
    targets = torch.arange(B)
    loss_i = F.cross_entropy(logits_i2t, targets, reduction="mean")
    loss_t = F.cross_entropy(logits_t2i, targets, reduction="mean")
    return 0.5 * (loss_i + loss_t)
    
    
    
# -----------------------------------------------------------------


# --------------------- REFERENCE + SELF-CHECK --------------------
def _ref_info_nce_clip(img, txt, tau=0.07):
    B = img.size(0)
    logits = (img @ txt.t()) / tau                # (B,B)
    targets = torch.arange(B, device=img.device)
    loss_i = F.cross_entropy(logits, targets, reduction="mean")
    loss_t = F.cross_entropy(logits.t(), targets, reduction="mean")
    return 0.5 * (loss_i + loss_t)

def _self_check():
    torch.manual_seed(0)
    B, D = 8, 64
    img = F.normalize(torch.randn(B, D), dim=-1, p=2).requires_grad_()
    txt = F.normalize(torch.randn(B, D), dim=-1, p=2)

    try:
        my_loss = info_nce_clip(img, txt)
    except NotImplementedError:
        print("🔧  Implement info_nce_clip and re-run the cell.")
        return

    ref_loss = _ref_info_nce_clip(img, txt)

    # Forward check
    assert torch.allclose(my_loss.detach(), ref_loss, atol=1e-6), \
        f"Loss mismatch: {my_loss} vs {ref_loss}"

    # Gradient check wrt image features
    ref_loss.backward()
    grad_ref = img.grad.clone()
    img.grad.zero_()
    my_loss.backward()
    grad_my = img.grad

    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch w.r.t. image features"

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


Problem 32 – Vectorized K-Means
==============================
Implement the classic **K-Means** clustering algorithm in PyTorch
without any Python loops over the data points or cluster indices.

Input
-----
* **data**  : `(N, D)` float32 tensor (no grad needed)
* **k**     : int  (2 ≤ k ≤ N)
* **num_iters** : int, number of full E→M iterations (default = 20)

Output
------
A tuple `(centroids, labels)` where

* **centroids** : `(k, D)` tensor, sorted by the first dimension
* **labels**    : `(N,)` long tensor giving the cluster index (0-based)
                  that each point belongs to.

Algorithm
---------
1. **Initialisation** – take the **first k** rows of `data` as the
   starting centroids (deterministic for unit tests).
2. Loop `num_iters` times  
   ‣ **E-step** : assign each point to its nearest centroid  
   ‣ **M-step** : recompute each centroid as the mean of its assigned
     points (guaranteed non-empty with the chosen initialisation).
3. Return final centroids (sorted by row index) and labels.

Implementation rules
--------------------
* You MAY loop over *iterations* (`for _ in range(num_iters):`)
  but **MUST NOT** loop over N or k.
* All heavy math must be done with tensor ops/broadcasting.

In [42]:
import torch
from typing import Tuple

# -------------------------- YOUR CODE ---------------------------
def kmeans(
    data: torch.Tensor,
    k: int,
    num_iters: int = 20
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Vectorised K-Means.

    Returns
    -------
    centroids : (k, D)  tensor
    labels    : (N,)    long tensor
    """
    
    N, D = data.shape
    centroids = data[:k].clone()
    
    for _ in range(num_iters):
        distance = torch.norm(data.view(N, 1, D) - centroids.view(1, k, D), dim=-1) # N, k
        _, labels = torch.min(distance, dim=-1)
        centroids.scatter_reduce_(dim=0, index=labels.view(N, 1).expand(N, k), src=data, reduce="mean", include_self=False)
        
    centroids, labels = deterministic_sort(centroids, labels)
    return centroids, labels
    
# -----------------------------------------------------------------

def deterministic_sort(centroids, labels):
    # sort centroids to make deterministic comparison easier
    order = centroids[:, 0].argsort()
    remap = order.argsort()
    centroids = centroids[order]
    labels = remap[labels]       # old labels to new labels
    return centroids, labels

# -------------------- Reference & Self-Check ---------------------
def _ref_kmeans(data: torch.Tensor, k: int, num_iters: int = 20):
    N, D = data.shape
    centroids = data[:k].clone()                       # (k,D)

    for _ in range(num_iters):
        # squared distances: (N,k)
        dists = ((data.unsqueeze(1) - centroids.unsqueeze(0)) ** 2).sum(-1)
        labels = dists.argmin(dim=1)                  # (N,)

        # recompute centroids
        eye = torch.eye(k, device=data.device)[labels]        # (N,k) one-hot
        counts = eye.sum(0)                                  # (k,)
        sums   = eye.T @ data                                # (k,D)
        centroids = sums / counts.unsqueeze(1)

    centroids, labels = deterministic_sort(centroids, labels)
    return centroids, labels


def _self_check():
    torch.manual_seed(0)
    # synthetic 2-cluster toy data
    N, D, k = 50, 2, 2
    data = torch.cat([torch.randn(N // 2, D) * 0.2 + torch.tensor([2, 0]),
                      torch.randn(N // 2, D) * 0.3 + torch.tensor([-2, 0])])

    try:
        cent_my, lbl_my = kmeans(data, k, num_iters=15)
    except NotImplementedError:
        print("🔧  Implement kmeans and re-run the cell.")
        return

    cent_ref, lbl_ref = _ref_kmeans(data, k, num_iters=15)

    # Compare centroids (order-sorted)
    assert torch.allclose(cent_my, cent_ref, atol=1e-3), \
        "Centroids differ from reference."

    # Compare cluster assignment consistency
    assert (lbl_my == lbl_ref).all(), "Labels differ from reference."

    print("✅  basic K-Means forward test passed")

_self_check()

✅  basic K-Means forward test passed


Problem 33 – Linear regression (no bias)
=======================================
Implement a *custom autograd* `Function` that returns the mean-squared
error (MSE) loss for a linear model **without an intercept**:

    ŷ = X @ w                 # X: (N, D), w: (D,)
    L  = ½ · mean( (ŷ − y)² )   (scalar)

Your task
---------
1. Fill in `LinearMSEFunction.forward` — compute the scalar loss.
2. Fill in `LinearMSEFunction.backward` — compute ∂L/∂w only
   (return `None` for the non-learnable inputs X and y).

Vectorised gradient formula
---------------------------
With the ½ scaling, the gradient is

    ∂L/∂w = (1/N) · Xᵀ · (ŷ − y)        # shape (D,)

No explicit loops over N or D.

In [48]:
import torch
from typing import Optional, Tuple

import torch
from typing import Tuple, Optional

# ----------------------------- YOUR CODE ------------------------
class LinearMSEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                w: torch.Tensor,    # (D,)  requires_grad
                X: torch.Tensor,    # (N,D)
                y: torch.Tensor):   # (N,)
        """
        Return scalar loss.  Save tensors needed for backward in ctx.
        """
        y_pred = X @ w
        loss = 0.5 * (y_pred - y).pow(2).mean()
        
        ctx.save_for_backward(w, X, y)
        
        return loss
        

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        grad_out is a scalar (shape []); propagate it to ∂L/∂w.
        Return grads w.r.t. (w, X, y) in that order.
        """
        w, X, y = ctx.saved_tensors
        
        N, D = X.shape
        
        return (X.T @ X @ w - X.T @ y) / N, None, None


def linear_mse_loss(w: torch.Tensor,
                    X: torch.Tensor,
                    y: torch.Tensor) -> torch.Tensor:
    """Convenience wrapper."""
    return LinearMSEFunction.apply(w, X, y)
# ----------------------------------------------------------------


# --------------------- Reference + Self-Check -------------------
def _ref_linear_mse(w, X, y):
    y_hat = X @ w
    return 0.5 * (y_hat - y).pow(2).mean()

def _self_check():
    torch.manual_seed(0)
    N, D = 13, 5
    X = torch.randn(N, D)
    y = torch.randn(N)
    w = torch.randn(D, requires_grad=True)

    try:
        loss_my = linear_mse_loss(w, X, y)
    except NotImplementedError:
        print("🔧  Fill in LinearMSEFunction forward/backward.")
        return

    loss_ref = _ref_linear_mse(w, X, y)

    # Forward value
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-6), \
        "Loss mismatch."

    # Gradient
    loss_ref.backward()
    grad_ref = w.grad.clone()
    w.grad.zero_()
    loss_my.backward()
    grad_my = w.grad
    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch."

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


Problem 34 – Binary logistic regression (no bias)
================================================
Implement a custom autograd `Function` that computes the *binary
cross-entropy* (BCE) loss for a sigmoid model without bias:

    p   = σ( X @ w )            # σ is sigmoid
    L   = mean(  − y·log p  − (1−y)·log(1−p) )    (scalar)

where y ∈ {0,1}.  No explicit loops over N or D.

Gradient formula
----------------
∂L/∂w = (1/N) · Xᵀ · (p − y)      # shape (D,)

In [54]:
import torch
import torch.nn.functional as F
from typing import Tuple, Optional

# ----------------------------- YOUR CODE ------------------------
class LogisticBCEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                w: torch.Tensor,   # (D,)  requires_grad
                X: torch.Tensor,   # (N,D)
                y: torch.Tensor):  # (N,)   0/1 floats
        """
        Save tensors needed for backward. Return scalar BCE loss.
        """
        p = torch.sigmoid(X @ w)
        loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()
        ctx.save_for_backward(X, y, p)
        return loss
        

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> Tuple[Optional[torch.Tensor], ...]:
        """
        Return grads for (w, X, y) in order.  Only w gets a gradient.
        """
        X, y, p = ctx.saved_tensors
        grad = -((y - p).view(-1, 1) * X).mean(dim=0) # N
        return grad, None, None


def logistic_bce_loss(w: torch.Tensor,
                      X: torch.Tensor,
                      y: torch.Tensor) -> torch.Tensor:
    return LogisticBCEFunction.apply(w, X, y)
# ----------------------------------------------------------------


# --------------------- Reference + Self-Check -------------------
def _ref_logistic_bce(w, X, y):
    p = torch.sigmoid(X @ w)
    return F.binary_cross_entropy(p, y, reduction="mean")

def _self_check():
    torch.manual_seed(1)
    N, D = 17, 4
    X = torch.randn(N, D)
    y = torch.randint(0, 2, (N,)).float()
    w = torch.randn(D, requires_grad=True)

    try:
        loss_my = logistic_bce_loss(w, X, y)
    except NotImplementedError:
        print("🔧  Fill in LogisticBCEFunction forward/backward.")
        return

    loss_ref = _ref_logistic_bce(w, X, y)

    # Forward check
    assert torch.allclose(loss_my.detach(), loss_ref, atol=1e-6), \
        "Loss mismatch."

    # Gradient check
    loss_ref.backward()
    grad_ref = w.grad.clone()
    w.grad.zero_()
    loss_my.backward()
    grad_my = w.grad
    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch."

    print("✅  basic forward & backward tests passed")

_self_check()

✅  basic forward & backward tests passed


Problem 35 – Soft-max Regression Loss with Manual Gradient
=========================================================
Write a custom autograd.Function that implements the mean cross-entropy
loss for *bias-free* multiclass logistic regression.

Definitions
-----------
Inputs
  • **W**  – weights, shape **(D, C)**, `requires_grad=True`
  • **X**  – features, shape **(N, D)**
  • **y**  – integer class labels, shape **(N,)**, values ∈ {0,…,C−1}

Logits
       Z = X · W                         # (N, C)

Soft-max probabilities
       P = softmax(Z, dim=1)             # (N, C)

Loss  (mean negative log-likelihood)
       L =  (1/N) · Σ_i  −log P[i, y_i]

Gradient w.r.t. W
-----------------
Let **Y_onehot** be the one-hot matrix of `y` (N×C).  
Then the analytical gradient is

       ∂L/∂W = (1/N) · Xᵀ · (P − Y_onehot)      # shape (D, C)

Your Task
~~~~~~~~~
Implement **both** `forward` and `backward` in `SoftmaxRegLossFn`
using the formula above.  
Gradients for `X` and `y` are **None** (they’re treated as constants).

In [1]:
import torch
import torch.nn.functional as F
from typing import Tuple, Optional

# ------------------------ YOUR CODE ------------------------
class SoftmaxRegLossFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx,
                W: torch.Tensor,          # (D, C)
                X: torch.Tensor,          # (N, D)
                y: torch.Tensor           # (N,)
                ) -> torch.Tensor:
        raise NotImplementedError          # <— fill this in

    @staticmethod
    def backward(ctx,
                 grad_out: torch.Tensor
                 ) -> Tuple[Optional[torch.Tensor], None, None]:
        raise NotImplementedError          # <— fill this in


def softmax_reg_loss(W: torch.Tensor,
                     X: torch.Tensor,
                     y: torch.Tensor) -> torch.Tensor:
    """Convenience wrapper around the custom autograd Function."""
    return SoftmaxRegLossFn.apply(W, X, y)
# -----------------------------------------------------------------


# --------------- Reference implementation & self-check -----------
def _ref_softmax_loss(W, X, y):
    logits = X @ W
    return F.cross_entropy(logits, y, reduction="mean")

def _self_check_softmax():
    torch.manual_seed(0)
    N, D, C = 40, 6, 5
    X = torch.randn(N, D)
    W = torch.randn(D, C, requires_grad=True)
    y = torch.randint(0, C, (N,))

    try:
        my_loss = softmax_reg_loss(W, X, y)
    except NotImplementedError:
        print("🔧  Implement SoftmaxRegLossFn forward/backward.")
        return

    ref_loss = _ref_softmax_loss(W, X, y)

    # Forward check
    assert torch.allclose(my_loss.detach(), ref_loss, atol=1e-6), \
        f"Loss mismatch: {my_loss} vs {ref_loss}"

    # Gradient check w.r.t. W
    ref_loss.backward()
    grad_ref = W.grad.clone()
    W.grad.zero_()
    my_loss.backward()
    grad_my = W.grad
    assert torch.allclose(grad_my, grad_ref, atol=1e-6), \
        "Gradient mismatch on W"

    print("✅  Multiclass logistic regression forward & backward OK")

_self_check_softmax()

🔧  Implement SoftmaxRegLossFn forward/backward.
